# Iteración, recursos y concurrencia

**Unidad 1 · Sesión 3**

Generadores, context managers y concurrencia con `asyncio`.

## Contenido

| Sección | Tema |
|---|---|
| 1 | Generadores y consumo bajo demanda |
| 2 | `with`, decoradores y `@contextmanager` |
| 3 | `async`/`await` y tareas concurrentes |
| 4 | Tiempos límite y limpieza |

Al terminar, podrás reconocer cuándo conviene procesar valores uno a uno,
mantener un recurso abierto solo durante su uso y solapar esperas independientes.

## 1. Generadores y consumo bajo demanda

Una lista conserva sus elementos. Un **iterador** mantiene la posición de un
recorrido; `next()` solicita el siguiente valor. Cuando se agota, no vuelve al
inicio. Una función con `yield` crea un generador: produce un valor cuando el
consumidor lo pide y continúa desde donde se quedó.

In [35]:
import tempfile
from collections import Counter

from prompt_toolkit.utils import to_int

values = iter([18.0, 27.0, 32.0])
#print(next(values))
#print(list(values))
#print(list(values))  # The iterator is exhausted.


def mi_flujo():
    for i in range(15):
        yield i  # Produce el dato y se pausa

# Se consume de forma perezosa
for dato in mi_flujo():
    print(dato)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [36]:
from collections.abc import Iterator


def readings(values: list[float]) -> Iterator[float]:
    for value in values:
        print(f"Reading {value}")
        yield value


stream = readings([18.0, 27.0, 32.0])
print("Generator created")
print(f"{next(stream)}\n")
print(list(stream))

Generator created
Reading 18.0
18.0

Reading 27.0
Reading 32.0
[27.0, 32.0]


Crear `stream` no ejecutó el cuerpo de `readings`. El primer `next()` produjo
un solo valor; `list(stream)` consumió el resto. Esto permite recorrer datos
sin construir antes una lista con todos los resultados. Si el consumidor usa
`list()`, entonces sí materializa lo que queda.

- [x] terminado
### <span style="color: red;"> Ejercicio 1 · Números pares bajo demanda</span>

Escribe `even_numbers(stop: int)`, una función generadora que produzca los
números pares desde cero hasta antes de `stop`. Comprueba que
`list(even_numbers(7))` sea `[0, 2, 4, 6]` y que `list(even_numbers(0))`
sea una lista vacía. Usa `yield`; no construyas una lista dentro de la función.

In [37]:
# Define even_numbers and check both cases.
def even_numbers(stop:int) -> Iterator[int]:
    for numero in range(0,stop):
        if numero % 2 == 0:
            yield numero
resultado = list(even_numbers(7))
lista_esperada = [0, 2, 4, 6]

assert resultado ==lista_esperada, "numeros pares incorrectos"

print("¡Assert pasado con éxito! La lista es:", resultado)

¡Assert pasado con éxito! La lista es: [0, 2, 4, 6]


## 2. Recursos con `with`

Un archivo debe cerrarse al terminar de usarlo. `with` delimita ese periodo:
al salir del bloque, Python llama al cierre del recurso, también si dentro
ocurre un error. El archivo se puede recorrer línea por línea.

In [38]:
from pathlib import Path
from tempfile import TemporaryDirectory


with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("north;18.0\nsouth;27.0\n", encoding="utf-8")

    with path.open(encoding="utf-8") as file:
        first_line = next(file).strip()
        print(first_line)

    assert file.closed

north;18.0


### Salida por error

El cierre no depende de que el bloque termine normalmente. El error puede
atenderse fuera de `with` y el archivo ya estará cerrado.

In [39]:
with TemporaryDirectory() as directory:
    path = Path(directory) / "readings.txt"
    path.write_text("invalid\n", encoding="utf-8")

    try:
        with path.open(encoding="utf-8") as file:
            raise ValueError("Invalid reading")
    except ValueError as error:
        print(error)

    assert file.closed

Invalid reading


### Qué es un decorador

Un decorador recibe una función y devuelve otra que añade comportamiento.
La forma `@announce` aplica `announce` a la función definida debajo. Aquí
solo observamos esa idea; no hace falta crear una biblioteca de decoradores.


In [40]:
def announce(function):
    def wrapper():
        print("Starting")
        return function()

    return wrapper


@announce
def show_message():
    print("Reading complete")


show_message()

Starting
Reading complete


### Crear un context manager con `@contextmanager`

`@contextmanager` permite escribir un context manager sin definir una clase.
La función llega hasta `yield` al entrar en `with`; el bloque `finally` se
ejecuta al salir. En esta demostración, el recurso es un archivo en memoria.

In [41]:
from contextlib import contextmanager
from io import StringIO


@contextmanager
def open_text(text: str) -> Iterator[StringIO]:
    file = StringIO(text)
    try:
        yield file
    finally:
        file.close()


with open_text("north\nsouth\n") as file:
    print(next(file).strip())

assert file.closed

north


- [x] terminado
### <span style="color: red;"> Ejercicio 2 · Cierre de un archivo</span>

Crea un archivo temporal con dos líneas. Ábrelo con `with`, lee solo la primera
y comprueba que esté cerrado después del bloque. Luego provoca un `ValueError`
dentro de otro bloque `with`, atiéndelo fuera del bloque y comprueba de nuevo
que el archivo quedó cerrado.

In [42]:
# Use TemporaryDirectory and Path; check file.closed in both situations.
import tempfile

with tempfile.TemporaryFile(mode="w+t") as file:
    file.writelines("linea 1 datos temporales \n linea 2 nuevos datos temporales")

    file.seek(0)
    line = file.readline()
    print(f"File:> {line}")

assert file.closed, "el archivo no se ha cerrado correctamente"
try:
    with tempfile.TemporaryFile(mode="xyz") as file:
        file.write("Más datos")
except ValueError as error:
    print(f"Value error: {error} no es posible realizar esto.")

File:> linea 1 datos temporales 

Value error: invalid mode: 'xyz' no es posible realizar esto.


## 3. Concurrencia con `asyncio`

Primero simulamos una lectura **síncrona**: `time.sleep` bloquea el hilo durante
los segundos indicados. Dos llamadas consecutivas de 3 y 2 segundos tardan
aproximadamente 5 segundos. Los mensajes permiten observar cuándo comienza y
termina cada operación.

In [43]:
import asyncio
import time
from time import perf_counter


def fetch_reading_sync(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    time.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
print(fetch_reading_sync("north", 3))
print(fetch_reading_sync("south", 2))
print(f"Synchronous: {perf_counter() - start:.1f} s")

Requesting north
Received north
('north', 18.0)
Requesting south
Received south
('south', 27.0)
Synchronous: 5.0 s


### La versión asíncrona

`async def` define una corrutina. `await asyncio.sleep()` suspende esa corrutina
y permite que otras tareas avancen. Cambiar solo `def` por `async def` no haría
que `time.sleep` dejara de bloquear: también necesitamos una espera compatible.

Dos `await` consecutivos siguen siendo **secuenciales**. En Jupyter y Colab usamos
`await` directamente en una celda.

In [44]:
async def fetch_reading(station: str, delay: float) -> tuple[str, float]:
    values = {"north": 18.0, "south": 27.0, "west": 32.0}
    print(f"Requesting {station}")
    await asyncio.sleep(delay)
    print(f"Received {station}")
    return station, values[station]


start = perf_counter()
north = await fetch_reading("north", 3)
south = await fetch_reading("south", 2)
print(north, south, f"Sequential: {perf_counter() - start:.1f} s")

Requesting north
Received north
Requesting south
Received south
('north', 18.0) ('south', 27.0) Sequential: 5.0 s


### Adaptar una función síncrona con `to_thread`

Si una biblioteca de E/S solo ofrece una función síncrona, `asyncio.to_thread`
puede ejecutarla en otro hilo. Este wrapper conserva la función original y
permite esperar su resultado con `await`. Pasamos la función **sin llamarla**,
seguida de sus argumentos. Un único `await` no hace que la lectura dure menos.

Esto sirve para esperas bloqueantes de E/S. No garantiza acelerar cálculos de
Python, y cancelar la espera no detiene automáticamente un hilo que ya trabaja.

In [45]:
async def fetch_reading_in_thread(station: str, delay: float) -> tuple[str, float]:
    return await asyncio.to_thread(fetch_reading_sync, station, delay)


print(await fetch_reading_in_thread("north", 2))

Requesting north
Received north
('north', 18.0)


### Una tarea en segundo plano con `create_task`

`create_task()` programa una corrutina para que avance mientras la corrutina
principal hace otro trabajo y cede el control. Conservamos la tarea en una
variable y al final usamos `await` para obtener su resultado y observar errores.
Aquí la lectura continúa durante la pausa de un segundo de la tarea principal.
No es un trabajo persistente: depende de que el event loop siga activo.

In [46]:
reading_task = asyncio.create_task(fetch_reading("north", 3))
print("Main task continues")
await asyncio.sleep(1)
print("Main task is ready to use the reading")
print(await reading_task)

Main task continues
Requesting north
Main task is ready to use the reading
Received north
('north', 18.0)


### Reunir resultados con `gather`

`gather()` programa las corrutinas y devuelve los resultados en el orden en que
las pasamos, aunque terminen en otro orden. Las esperas de 3 y 2 segundos se
solapan: el total se acerca a 3 segundos.

In [47]:
start = perf_counter()
north, south = await asyncio.gather(
    fetch_reading("north", 3),
    fetch_reading("south", 2),
)
print(north, south, f"Gather: {perf_counter() - start:.1f} s")

Requesting north
Requesting south
Received south
Received north
('north', 18.0) ('south', 27.0) Gather: 3.0 s


### Delimitar las tareas con `TaskGroup`

`TaskGroup` espera sus tareas al salir de `async with`. Conservamos cada objeto
tarea para consultar después su resultado. Si una tarea falla con una excepción
que no es de cancelación, el grupo cancela las restantes y espera su limpieza.

En cambio, `gather` con sus opciones predeterminadas propaga la primera excepción,
pero las otras tareas pueden continuar. Aquí ambos ejemplos tienen éxito; la
diferencia está en cómo organizan su finalización y sus errores.

In [48]:
start = perf_counter()
async with asyncio.TaskGroup() as group:
    north_task = group.create_task(fetch_reading("north", 3))
    south_task = group.create_task(fetch_reading("south", 2))
print(north_task.result(), south_task.result())
print(f"TaskGroup: {perf_counter() - start:.1f} s")

Requesting north
Requesting south
Received south
Received north
('north', 18.0) ('south', 27.0)
TaskGroup: 3.0 s


Los tiempos son aproximados. Observa que ambas solicitudes empiezan antes de
que llegue la primera respuesta. La concurrencia permite solapar esperas
independientes; no acelera por sí sola un cálculo intensivo.

- [x] terminado
### <span style="color: red;">Ejercicio 3 · Dos lecturas concurrentes</span>

Crea dos tareas con `TaskGroup`: `fetch_reading("north", 3)` y
`fetch_reading("west", 1)`.
Después del bloque:
- comprueba con `assert`los dos resultados.
- Explica por qué `west` puede anunciarse primero aunque la tarea de `north` se haya creado antes.

In [54]:
# Create two tasks, read their results after TaskGroup, and check both tuples.
async  with asyncio.TaskGroup() as group:
    north_task = group.create_task(fetch_reading("north", 3))
    west_task = group.create_task(fetch_reading("west", 1))

north_result = north_task.result()
west_result = west_task.result()

print(north_task.result(), west_task.result())
print(f"{perf_counter() - start:.1f} s")

assert north_result == ("north", 18.0), f"Esperado ('north', 18.0), pero se obtuvo {north_result}"
assert west_result == ("west", 32.0), f"Esperado ('west', 32.0), pero se obtuvo {west_result}"

#la razon por la que west regresa primero que north es por que el delay configurado es de 1 segundo
#cuando se recibe en:  await asyncio.sleep(delay)

Requesting north
Requesting west
Received west
Received north
('north', 18.0) ('west', 32.0)
116.5 s


## 4. Tiempo límite y limpieza

Una operación puede tardar demasiado. `asyncio.timeout()` establece un tiempo
máximo para el bloque. Si se agota, se produce `TimeoutError`. La corrutina
puede usar `finally` para limpiar su estado incluso cuando se cancela la espera.

In [50]:
events: list[str] = []


async def wait_for_update() -> None:
    events.append("started")
    try:
        await asyncio.Event().wait()
    finally:
        events.append("cleaned up")


try:
    async with asyncio.timeout(2):
        await wait_for_update()
except TimeoutError:
    print("Deadline reached")

assert events == ["started", "cleaned up"]
print(events)

Deadline reached
['started', 'cleaned up']


- [x] terminado
### <span style="color: red;">Ejercicio 4 · Comprobar la limpieza</span>

Escribe una corrutina
- que añada `"started"` a una lista,
- espere 3 segundos
- y añada `"cleaned up"` en `finally`.
- Llámala dentro de `asyncio.timeout(1)`,
- atiende `TimeoutError`
- y comprueba que ambos mensajes quedaron en la lista.
- Después prueba una espera de 1 segundo con un límite de 3 segundos.

In [57]:
# Define a coroutine with try/finally; check a timeout and a completed wait.
events: list[str] = []

async def wait_func(sleep_time: float) -> None:
    events.append("started")
    try:
        await asyncio.sleep(sleep_time)
    finally:
        events.append("cleaned up")

events.clear()

try:
    async with asyncio.timeout(1):
        await wait_func(sleep_time=3)
except TimeoutError:
    print("Prueba 1: Tiempo límite alcanzado (TimeoutError capturado correctamente).")

assert events == ["started", "cleaned up"], f"Se esperaba ['started', 'cleaned up'], pero se obtuvo {events}"
print(f"Estado de la lista en Prueba 1: {events}\n")

events.clear()

try:
    async with asyncio.timeout(3):
        await wait_func(sleep_time=1)
except TimeoutError:
    print("Prueba 2: Esto no debería ejecutarse.")

assert events == ["started", "cleaned up"], f"Se esperaba ['started', 'cleaned up'], pero se obtuvo {events}"
print(f"Estado de la lista en Prueba 2: {events}")



Prueba 1: Tiempo límite alcanzado (TimeoutError capturado correctamente).
Estado de la lista en Prueba 1: ['started', 'cleaned up']

Estado de la lista en Prueba 2: ['started', 'cleaned up']


1. **¿Por qué `finally` funciona en la Prueba 1?**<br>Cuando `asyncio.timeout(1)` interrumpe `wait_func` al cumplirse 1 segundo, Python lanza internamente
una excepción de cancelación (`CancelledError`) a la corrutina[cite: 2].
El bloque `finally` atrapa ese momento antes de que la función se destruya y
 asegura que `"cleaned up"` sea añadido a la lista.

2. **Reutilización del parámetro `sleep_time`:**<br>Al parametrizar el tiempo de `asyncio.sleep(sleep_time)`, podemos usar la misma corrutina para verificar
tanto el caso donde el límite falla (3s de espera vs 1s límite) como el caso donde tiene éxito
(1s de espera vs 3s límite).

## Cierre

- Un generador produce valores cuando se solicitan; consumirlo por completo
  puede volver a ocupar memoria para guardar los resultados.
- `with` delimita el uso de un recurso; `@contextmanager` permite definir
  esa entrada y salida mediante una función.
- `asyncio` ayuda a solapar esperas independientes. Usa un tiempo límite
  cuando una espera no deba continuar indefinidamente.

En un archivo `.py` se inicia el event loop desde una función principal con
`asyncio.run(main())`; en esta notebook usamos `await` directamente.

## Referencias

- [Tutorial de Python: generadores](https://docs.python.org/3/tutorial/classes.html#generators).
- [Python: `contextlib`](https://docs.python.org/3/library/contextlib.html).
- [Python: corrutinas y tareas](https://docs.python.org/3/library/asyncio-task.html).
- [Python: ejecutar `asyncio`](https://docs.python.org/3/library/asyncio-runner.html).

In [58]:
#patron observer con async await

import asyncio
from typing import Protocol

# 1. Definimos la interfaz del Observador usando Protocols (Tipado estructural con Mypy)
class Observer(Protocol):
    async def update(self, order_id: str, total: float) -> None:
        ...

# 2. Implementamos los Observadores Asíncronos
class EmailService:
    async def update(self, order_id: str, total: float) -> None:
        await asyncio.sleep(1)  # Simulamos una petición de red lenta
        print(f"📧 [Email] Correo enviado para la orden {order_id}")

class AnalyticsService:
    async def update(self, order_id: str, total: float) -> None:
        await asyncio.sleep(0.5)  # Simulamos guardar en base de datos
        print(f"📊 [Analytics] Registro guardado para la orden {order_id} por ${total}")

# 3. El Sujeto (Subject) que gestiona los eventos de forma asíncrona
class OrderSystem:
    def __init__(self) -> None:
        self._observers: list[Observer] = []

    def attach(self, observer: Observer) -> None:
        self._observers.append(observer)

    async def notify(self, order_id: str, total: float) -> None:
        # Lanzamos todas las notificaciones en paralelo usando asyncio.gather
        tasks = [observer.update(order_id, total) for observer in self._observers]
        await asyncio.gather(*tasks)

    async def create_order(self, order_id: str, total: float) -> None:
        print(f"🛍️ [Sistema] Creando orden {order_id}...")
        # Lógica de negocio para crear la orden...
        await self.notify(order_id, total)

# --- Ejecución del código ---
async def main() -> None:
    order_system = OrderSystem()

    # Registramos observadores
    order_system.attach(EmailService())
    order_system.attach(AnalyticsService())

    # Ejecutamos la acción asíncrona
    await order_system.create_order("ORD-1234", 150.50)

# Para correrlo en un script tradicional: asyncio.run(main())
await main()

🛍️ [Sistema] Creando orden ORD-1234...
📊 [Analytics] Registro guardado para la orden ORD-1234 por $150.5
📧 [Email] Correo enviado para la orden ORD-1234
